In [2]:
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf

RAW = Path("../raw_data")
SR = 16000

def get_xy(split):
    """One row per conversation in raw_data/<split>/ with its two audio files."""
    rows = []
    for conv in sorted(p for p in (RAW / split).iterdir() if p.is_dir()):
        a, b = conv / "speaker_a.flac", conv / "speaker_b.flac"
        if not (a.exists() and b.exists()):
            print(f"  ⚠ skipping {conv.name}: missing audio"); continue
        rows.append({"conversation_id": conv.name, "speaker_a": a, "speaker_b": b})
    df = pd.DataFrame(rows)
    print(f"{split:<5}: {len(df):>5} conversations")
    return df, df           # X and y come from the same two files (see load_x / load_y)

def load_y(row, start_sec=0.0, dur_sec=None):
    """y: the two separate tracks -> shape (2, samples). Whole file if dur_sec is None."""
    start = int(start_sec * SR)
    frames = -1 if dur_sec is None else int(dur_sec * SR)
    a, _ = sf.read(row.speaker_a, start=start, frames=frames, dtype="float32")
    b, _ = sf.read(row.speaker_b, start=start, frames=frames, dtype="float32")
    n = max(len(a), len(b))
    return np.stack([np.pad(a, (0, n - len(a))), np.pad(b, (0, n - len(b)))])

def load_x(row, start_sec=0.0, dur_sec=None):
    """X: both speakers fused into one track -> shape (samples,)."""
    return load_y(row, start_sec, dur_sec).sum(axis=0)

X_train, y_train = get_xy("train")
X_val,   y_val   = get_xy("val")
X_test,  y_test  = get_xy("test")

train:  1090 conversations
val  :   162 conversations
test :   243 conversations


In [49]:
X_train.info

<bound method DataFrame.info of         conversation_id                                          speaker_a  \
0     conv_0084a98344ee  ../raw_data/train/conv_0084a98344ee/speaker_a....   
1     conv_009afb2f7521  ../raw_data/train/conv_009afb2f7521/speaker_a....   
2     conv_014abf8ecbc5  ../raw_data/train/conv_014abf8ecbc5/speaker_a....   
3     conv_016ff57f47cc  ../raw_data/train/conv_016ff57f47cc/speaker_a....   
4     conv_021eeddb3616  ../raw_data/train/conv_021eeddb3616/speaker_a....   
...                 ...                                                ...   
1085  conv_ff4084582ad6  ../raw_data/train/conv_ff4084582ad6/speaker_a....   
1086  conv_ff9cdf08670e  ../raw_data/train/conv_ff9cdf08670e/speaker_a....   
1087  conv_ffd182ffbb59  ../raw_data/train/conv_ffd182ffbb59/speaker_a....   
1088  conv_ffee75c9552d  ../raw_data/train/conv_ffee75c9552d/speaker_a....   
1089  conv_fffdfa8e6228  ../raw_data/train/conv_fffdfa8e6228/speaker_a....   

                               

In [50]:
y_train.info

<bound method DataFrame.info of         conversation_id                                          speaker_a  \
0     conv_0084a98344ee  ../raw_data/train/conv_0084a98344ee/speaker_a....   
1     conv_009afb2f7521  ../raw_data/train/conv_009afb2f7521/speaker_a....   
2     conv_014abf8ecbc5  ../raw_data/train/conv_014abf8ecbc5/speaker_a....   
3     conv_016ff57f47cc  ../raw_data/train/conv_016ff57f47cc/speaker_a....   
4     conv_021eeddb3616  ../raw_data/train/conv_021eeddb3616/speaker_a....   
...                 ...                                                ...   
1085  conv_ff4084582ad6  ../raw_data/train/conv_ff4084582ad6/speaker_a....   
1086  conv_ff9cdf08670e  ../raw_data/train/conv_ff9cdf08670e/speaker_a....   
1087  conv_ffd182ffbb59  ../raw_data/train/conv_ffd182ffbb59/speaker_a....   
1088  conv_ffee75c9552d  ../raw_data/train/conv_ffee75c9552d/speaker_a....   
1089  conv_fffdfa8e6228  ../raw_data/train/conv_fffdfa8e6228/speaker_a....   

                               

In [51]:
row = X_train.iloc[0]
x = load_x(row, start_sec=10, dur_sec=4)   # (64000,)    the fused input
y = load_y(row, start_sec=10, dur_sec=4)   # (2, 64000)  [speaker_a, speaker_b] targets

In [56]:
x.shape

(64000,)

In [57]:
y.shape

(2, 64000)

In [60]:
from pathlib import Path
import numpy as np, soundfile as sf
from tqdm.auto import tqdm

RAW = Path("../raw_data")
SR = 16000

def fuse(conv):
    """Mix speaker_a + speaker_b into conv/fused.flac (same method as fuse.ipynb)."""
    out = conv / "fused.flac"
    if out.exists():                                   # already fused -> skip
        return out
    with sf.SoundFile(conv / "speaker_a.flac") as fa, sf.SoundFile(conv / "speaker_b.flac") as fb, \
         sf.SoundFile(out, "w", samplerate=fa.samplerate, channels=1, subtype="PCM_24") as fo:
        length = min(len(fa), len(fb))
        block = 60 * fa.samplerate                     # 60 s at a time, keeps RAM low
        for start in range(0, length, block):
            n = min(block, length - start)
            a = fa.read(n, dtype="float32", always_2d=True).mean(axis=1)
            b = fb.read(n, dtype="float32", always_2d=True).mean(axis=1)
            fo.write((a + b) / 2)
    return out

def get_xy(split):
    """X = fused files, y = (speaker_a, speaker_b) files, same order."""
    convs = sorted(p for p in (RAW / split).iterdir() if p.is_dir())
    X = [fuse(c) for c in tqdm(convs, desc=f"fusing {split}")]
    y = [(c / "speaker_a.flac", c / "speaker_b.flac") for c in convs]
    return X, y

X_train, y_train = get_xy("train")
X_val,   y_val   = get_xy("val")
X_test,  y_test  = get_xy("test")

/Users/annettesachs/code/asachs25/Sonara/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
fusing test: 100%|██████████| 243/243 [03:46<00:00,  1.07it/s]
